# Download AIDev Dataset

This notebook downloads the AIDev dataset from Hugging Face for the MSR 2026 Mining Challenge.

The dataset contains:
- 33,596 curated Agentic-PRs from 2,807 popular repositories
- Data from 5 AI agents: Claude Code, Cursor, Devin, GitHub Copilot, OpenAI Codex
- PR metadata, commits, comments, reviews, and file-level changes

In [1]:
import pandas as pd
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

In [2]:
# Set up data directory
data_dir = Path("../data")
data_dir.mkdir(exist_ok=True)
print(f"Data directory: {data_dir.absolute()}")

Data directory: /Users/lukas.twist/code/agent-library-usage/notebooks/../data


## Download Main Tables

We'll download the main parquet files from the AIDev dataset.

In [3]:
# Download the main tables
print("Downloading all_repository.parquet...")
repo_df = pd.read_parquet("hf://datasets/hao-li/AIDev/all_repository.parquet")
print(f"Repositories: {len(repo_df):,} rows")
print(f"Columns: {list(repo_df.columns)}")
repo_df.head()

Repositories: 116,211 rows
Columns: ['id', 'url', 'license', 'full_name', 'language', 'forks', 'stars']


,id,url,license,full_name,language,forks,stars
0,987641962,https://api.github.com/repos/1010-dev/senjudev...,None,1010-dev/senjudev-site,TypeScript,1.0,0.0
1,990249393,https://api.github.com/repos/106-/HellSinkerWa...,None,106-/HellSinkerWallPaper,Java,0.0,0.0
2,1009549206,https://api.github.com/repos/1genadam/tileshop...,None,1genadam/tileshop-rag,Python,0.0,0.0
3,983546765,https://api.github.com/repos/1kimnet/ETL-pipeline,None,1kimnet/ETL-pipeline,Python,0.0,0.0
4,1024190983,https://api.github.com/repos/20m61/lightningta...,None,20m61/lightningtalk-circle,JavaScript,0.0,0.0


In [4]:
print("\nDownloading all_pull_request.parquet...")
pr_df = pd.read_parquet("hf://datasets/hao-li/AIDev/all_pull_request.parquet")
print(f"Pull Requests: {len(pr_df):,} rows")
print(f"Columns: {list(pr_df.columns)}")
pr_df.head()

Pull Requests: 932,791 rows
Columns: ['id', 'number', 'title', 'body', 'agent', 'user_id', 'user', 'state', 'created_at', 'closed_at', 'merged_at', 'repo_id', 'repo_url', 'html_url']


,id,number,title,body,agent,user_id,user,state,created_at,closed_at,merged_at,repo_id,repo_url,html_url
0,3264016139,1688,`metta code` --> `metta clip` and additional p...,Remove unused `root_key` variable to fix ruff ...,Claude_Code,37011,jacklionheart,closed,2025-07-25T18:15:36Z,2025-07-25T19:17:23Z,2025-07-25T19:17:23Z,8.439884e+08,https://api.github.com/repos/Metta-AI/metta,https://github.com/Metta-AI/metta/pull/1688
1,3264021033,41,feat: Comprehensive ruff error resolution with...,## 🎯 Mission Accomplished: 100% Ruff Error Res...,Claude_Code,131842369,Draco3310,open,2025-07-25T18:17:57Z,None,None,9.920635e+08,https://api.github.com/repos/Draco3310/Gal-Fri...,https://github.com/Draco3310/Gal-Friday2/pull/41
2,3264042289,1600,Add Evals frontend implementation plan and HTM...,\nCreate comprehensive implementation plan for...,Claude_Code,6766889,justicart,closed,2025-07-25T18:26:15Z,2025-07-25T23:19:14Z,None,9.267118e+08,https://api.github.com/repos/bolt-foundry/bolt...,https://github.com/bolt-foundry/bolt-foundry/p...
3,3264042318,1601,Add 4 new BfDs components for Evals interface ...,\nPhase 1 component creation for the Evals fro...,Claude_Code,6766889,justicart,closed,2025-07-25T18:26:16Z,2025-07-25T23:19:11Z,None,9.267118e+08,https://api.github.com/repos/bolt-foundry/bolt...,https://github.com/bolt-foundry/bolt-foundry/p...
4,3264067496,3,🚀 Complete Frontend-Backend API Integration wi...,## 🎯 Summary\n\nThis PR completes the **fronte...,Claude_Code,42357482,twitchyvr,closed,2025-07-25T18:39:14Z,2025-07-25T18:48:47Z,2025-07-25T18:48:47Z,1.025871e+09,https://api.github.com/repos/twitchyvr/Spaghetti,https://github.com/twitchyvr/Spaghetti/pull/3


In [5]:
print("\nDownloading all_user.parquet...")
user_df = pd.read_parquet("hf://datasets/hao-li/AIDev/all_user.parquet")
print(f"Users: {len(user_df):,} rows")
print(f"Columns: {list(user_df.columns)}")
user_df.head()

Users: 72,189 rows
Columns: ['id', 'login', 'followers', 'following', 'created_at']


,id,login,followers,following,created_at
0,149159513.0,00012122Cs,0.0,0.0,2023-10-27T10:22:50Z
1,86906973.0,000Sean000,0.0,2.0,2021-07-04T07:05:40Z
2,36679210.0,000alen,46.0,106.0,2018-02-20T21:00:55Z
3,201261210.0,000qhrey,0.0,1.0,2025-02-28T16:53:21Z
4,22735204.0,00125495,0.0,1.0,2016-10-10T01:24:34Z


## Save to Local Files

In [6]:
# Save to local parquet files
repo_df.to_parquet(data_dir / "all_repository.parquet", index=False)
pr_df.to_parquet(data_dir / "all_pull_request.parquet", index=False)
user_df.to_parquet(data_dir / "all_user.parquet", index=False)
print("\nSaved all files to data/ directory")


Saved all files to data/ directory


## Download AIDev-pop Tables

Now let's download the AIDev-pop subset tables (repos with 100+ stars) which include commit details.

In [7]:
# Download PR commits with file-level changes
print("\nDownloading pr_commit_details.parquet...")
commit_details_df = pd.read_parquet(
    "hf://datasets/hao-li/AIDev/pr_commit_details.parquet"
)
print(f"Commit Details: {len(commit_details_df):,} rows")
print(f"Columns: {list(commit_details_df.columns)}")
commit_details_df.head()

Commit Details: 711,923 rows
Columns: ['sha', 'pr_id', 'author', 'committer', 'message', 'commit_stats_total', 'commit_stats_additions', 'commit_stats_deletions', 'filename', 'status', 'additions', 'deletions', 'changes', 'patch']


,sha,pr_id,author,committer,message,commit_stats_total,commit_stats_additions,commit_stats_deletions,filename,status,additions,deletions,changes,patch
0,2f9d54dda4f0c87c19e0bbeb9936f525d0587e16,3271196926,devin-ai-integration[bot],devin-ai-integration[bot],Add llms.txt compilation system for AI model d...,23008,23008,0,.github/workflows/compile-llms-txt.yml,added,38.0,0.0,38.0,"@@ -0,0 +1,38 @@\n+name: Compile llms.txt\n+\n..."
1,2f9d54dda4f0c87c19e0bbeb9936f525d0587e16,3271196926,devin-ai-integration[bot],devin-ai-integration[bot],Add llms.txt compilation system for AI model d...,23008,23008,0,docs/compile_llms_txt.py,added,47.0,0.0,47.0,"@@ -0,0 +1,47 @@\n+import os\n+from pathlib im..."
2,2f9d54dda4f0c87c19e0bbeb9936f525d0587e16,3271196926,devin-ai-integration[bot],devin-ai-integration[bot],Add llms.txt compilation system for AI model d...,23008,23008,0,llms.txt,added,22923.0,0.0,22923.0,None
3,dbd1b5f129f7cffa5ce284d7255814c98bcc38a2,3271196926,devin-ai-integration[bot],devin-ai-integration[bot],Fix lint issues: remove unused variable and ap...,35,18,17,docs/compile_llms_txt.py,modified,18.0,17.0,35.0,"@@ -1,47 +1,48 @@\n import os\n from pathlib i..."
4,c2659cfdedf666c8f14753d71664563c2a932b23,3271196926,devin-ai-integration[bot],devin-ai-integration[bot],Update llms.txt to follow official standard wi...,23035,89,22946,docs/compile_llms_txt.py,modified,51.0,36.0,87.0,"@@ -3,45 +3,60 @@\n \n \n def compile_llms_txt..."


In [8]:
# Save commit details
commit_details_df.to_parquet(data_dir / "pr_commit_details.parquet", index=False)
print("Saved pr_commit_details.parquet")

Saved pr_commit_details.parquet


In [9]:
# Download the pull_request table (AIDev-pop subset)
print("\nDownloading pull_request.parquet (AIDev-pop)...")
pr_pop_df = pd.read_parquet("hf://datasets/hao-li/AIDev/pull_request.parquet")
print(f"Pull Requests (pop): {len(pr_pop_df):,} rows")
print(f"Columns: {list(pr_pop_df.columns)}")
pr_pop_df.to_parquet(data_dir / "pull_request.parquet", index=False)
pr_pop_df.head()

Pull Requests (pop): 33,596 rows
Columns: ['id', 'number', 'title', 'body', 'agent', 'user_id', 'user', 'state', 'created_at', 'closed_at', 'merged_at', 'repo_id', 'repo_url', 'html_url']


,id,number,title,body,agent,user_id,user,state,created_at,closed_at,merged_at,repo_id,repo_url,html_url
0,3264933329,2911,Fix: Wait for all partitions in load_collectio...,## Summary\n\nFixes an issue where `load_colle...,Claude_Code,108661493,weiliu1031,closed,2025-07-26T02:59:01Z,2025-07-29T07:01:20Z,None,191751505,https://api.github.com/repos/milvus-io/pymilvus,https://github.com/milvus-io/pymilvus/pull/2911
1,3265118634,2,ファイルパス参照を相対パスに統一し、doc/からdocs/に統一,## 背景\n\n現在、本プロジェクトにおいて以下のパス構成の不整合が生じています：\n\n...,Claude_Code,61827001,cm-kojimat,closed,2025-07-26T04:56:55Z,2025-07-26T22:12:24Z,2025-07-26T22:12:24Z,1025472321,https://api.github.com/repos/classmethod/tsumiki,https://github.com/classmethod/tsumiki/pull/2
2,3265640341,30,Add build staleness detection for debug CLI,## Summary\r\n\r\n Implements comprehensive b...,Claude_Code,7475,MSch,closed,2025-07-26T13:31:19Z,2025-07-26T13:37:22Z,2025-07-26T13:37:22Z,988488798,https://api.github.com/repos/steipete/Peekaboo,https://github.com/steipete/Peekaboo/pull/30
3,3265709660,205,feat: add comprehensive README screenshots wit...,## Type of Change\n\n- [ ] 🐛 `bug` - Bug fix (...,Claude_Code,80381,sugyan,closed,2025-07-26T14:07:22Z,2025-07-26T14:45:30Z,2025-07-26T14:45:30Z,999285986,https://api.github.com/repos/sugyan/claude-cod...,https://github.com/sugyan/claude-code-webui/pu...
4,3265782173,17625,chore: remove HashedPostStateProvider trait,## Summary\r\n\r\n#17545 \r\n\r\nRemove the un...,Claude_Code,47593288,adust09,open,2025-07-26T15:02:48Z,None,None,537233603,https://api.github.com/repos/paradigmxyz/reth,https://github.com/paradigmxyz/reth/pull/17625


In [10]:
# Download repository table (AIDev-pop)
print("\nDownloading repository.parquet (AIDev-pop)...")
repo_pop_df = pd.read_parquet("hf://datasets/hao-li/AIDev/repository.parquet")
print(f"Repositories (pop): {len(repo_pop_df):,} rows")
print(f"Columns: {list(repo_pop_df.columns)}")
repo_pop_df.to_parquet(data_dir / "repository.parquet", index=False)
repo_pop_df.head()

Repositories (pop): 2,807 rows
Columns: ['id', 'url', 'license', 'full_name', 'language', 'forks', 'stars']


,id,url,license,full_name,language,forks,stars
0,966235850,https://api.github.com/repos/kizuna-ai-lab/sokuji,AGPL-3.0,kizuna-ai-lab/sokuji,TypeScript,11,254
1,386644013,https://api.github.com/repos/freenet/freenet-core,NOASSERTION,freenet/freenet-core,Rust,94,2391
2,977155585,https://api.github.com/repos/coleam00/mcp-craw...,MIT,coleam00/mcp-crawl4ai-rag,Python,474,1447
3,528194129,https://api.github.com/repos/vexxhost/atmosphere,None,vexxhost/atmosphere,Smarty,34,142
4,703998226,https://api.github.com/repos/JonasKruckenberg/k23,Apache-2.0,JonasKruckenberg/k23,Rust,32,515


## Summary

In [11]:
print("\n" + "=" * 60)
print("DATASET DOWNLOAD COMPLETE")
print("=" * 60)
print(f"\nAll repositories: {len(repo_df):,}")
print(f"Popular repositories (100+ stars): {len(repo_pop_df):,}")
print(f"\nAll pull requests: {len(pr_df):,}")
print(f"Popular pull requests: {len(pr_pop_df):,}")
print(f"\nCommit details: {len(commit_details_df):,}")
print(f"Users: {len(user_df):,}")
print("\nFiles saved to:", data_dir.absolute())


DATASET DOWNLOAD COMPLETE

All repositories: 116,211
Popular repositories (100+ stars): 2,807

All pull requests: 932,791
Popular pull requests: 33,596

Commit details: 711,923
Users: 72,189

Files saved to: /Users/lukas.twist/code/agent-library-usage/notebooks/../data
